# 9 - Testing: Do the Models Perform Reliably?

**Covers:** Section 4.9.2 in full -- seed sensitivity, cross-validation stability, hyperparameter robustness, cross-algorithm agreement.

The single most important check is the first one. Section 4.9.2: *"If the Control-versus-Alignment-Augmented difference is smaller than the variation induced by changing the seed alone, that difference cannot be attributed to the feature, and this is reported as such."* A ΔR2 whose bootstrap interval excludes zero but which is smaller than seed noise is **not** a result.

**This is the slowest notebook**: 5 seeds x 2 models, plus 5 CV folds x 2 models, plus 3 hyperparameter settings x 2 models -- 26 XGBoost fits. Each section is independent; run them separately if needed.

**Produces:** `artifacts/robustness_seeds.csv`, `artifacts/robustness_cv.csv`, `artifacts/robustness_hyperparams.csv`, `artifacts/robustness_summary.json`.

In [ ]:
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold

sys.path.insert(0, os.path.abspath('.'))
import modeling_config as mc

pd.set_option('display.width', 170)
N_JOBS = -1
PAIR = ['control', 'alignment_augmented']

df = mc.load_corpus()
df['artist_list'] = mc.artist_lists(df['artist_ids'])
df['primary_artist'] = df['artist_list'].str[0]
splits = pd.read_csv(mc.artifact('splits.csv'))
config_xgb = mc.load_json('config_xgb.json')
central = mc.load_json('central_comparison.json')

df = df.merge(splits[['id', 'partition']], on='id', how='inner')
modelled = df[df['partition'].isin(['train', 'val', 'test'])].reset_index(drop=True)

X_all, blocks = mc.build_feature_frame(modelled)
mc.assert_clean(X_all)
y = modelled[mc.TARGET].astype(float)
part = modelled['partition']
cols_of = {name: mc.columns_for(name, blocks) for name in PAIR}

def fit_xgb(Xtr, ytr, Xva, yva, params=None, seed=mc.RANDOM_SEED):
    p = dict(config_xgb['params'])
    p.update(params or {})
    p['random_state'] = seed
    model = xgb.XGBRegressor(eval_metric='rmse', early_stopping_rounds=50,
                             n_jobs=N_JOBS, **p)
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return model

print('observed ΔR2 on the held-out test set (XGBoost): %+.5f'
      % central['xgboost']['delta_r2'])

## 1. Seed sensitivity

Retrain the central pair under five seeds on the same splits. Report the spread in R2, RMSE and MAE, and compare ΔR2 against the seed-induced standard deviation.

In [ ]:
SEEDS = [42, 7, 1337, 2024, 99]
tr, va, te = (part == 'train').to_numpy(), (part == 'val').to_numpy(), (part == 'test').to_numpy()

rows = []
for seed in SEEDS:
    per_seed = {}
    for name in PAIR:
        c = cols_of[name]
        t0 = time.time()
        model = fit_xgb(X_all.loc[tr, c], y[tr], X_all.loc[va, c], y[va], seed=seed)
        m = mc.regression_metrics(y[te], model.predict(X_all.loc[te, c]))
        per_seed[name] = m
        rows.append({'seed': seed, 'model': name, 'seconds': round(time.time() - t0, 1), **m})
        print('seed %-5d %-22s R2 %+.5f  RMSE %.4f  (%.1fs)'
              % (seed, name, m['r2'], m['rmse'], rows[-1]['seconds']))
    rows.append({'seed': seed, 'model': 'DELTA', 'seconds': 0.0,
                 'r2': per_seed['alignment_augmented']['r2'] - per_seed['control']['r2'],
                 'rmse': per_seed['alignment_augmented']['rmse'] - per_seed['control']['rmse'],
                 'mae': per_seed['alignment_augmented']['mae'] - per_seed['control']['mae']})

seeds_df = pd.DataFrame(rows)
seeds_df.to_csv(mc.artifact('robustness_seeds.csv'), index=False)
print()
print(seeds_df.pivot_table(index='seed', columns='model', values='r2').round(5).to_string())

In [ ]:
# The decisive comparison in Section 4.9.2.
control_spread = seeds_df.loc[seeds_df['model'] == 'control', 'r2']
aug_spread = seeds_df.loc[seeds_df['model'] == 'alignment_augmented', 'r2']
delta_spread = seeds_df.loc[seeds_df['model'] == 'DELTA', 'r2']

seed_sd = float(pd.concat([control_spread, aug_spread]).std())
mean_delta = float(delta_spread.mean())

print('R2 std across seeds (both models pooled): %.5f' % seed_sd)
print('mean ΔR2 across seeds:                    %+.5f' % mean_delta)
print('ΔR2 range across seeds:                   %+.5f to %+.5f'
      % (delta_spread.min(), delta_spread.max()))
print('ΔR2 sign consistent across all seeds:     %s'
      % bool(np.all(np.sign(delta_spread) == np.sign(mean_delta))))
print()
if abs(mean_delta) > seed_sd:
    print('PASS: ΔR2 exceeds the variation induced by changing the seed alone.')
else:
    print('FAIL: ΔR2 is smaller than seed-induced variation. Section 4.9.2 requires')
    print('      this be reported -- the difference cannot be attributed to the')
    print('      feature, regardless of what the bootstrap interval says.')

## 2. Cross-validation stability (artist-aware k-fold)

The bootstrap interval resamples a *single* split and so cannot detect split-specific effects. This refits the pair under artist-aware 5-fold CV, applying the same Option C edge cut inside every fold, and reports ΔR2 per fold.

In [ ]:
K = 5
sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=mc.RANDOM_SEED)
artist_lists = modelled['artist_list']

rows = []
for fold, (train_idx, hold_idx) in enumerate(
        sgkf.split(modelled, y=modelled['genre'], groups=modelled['primary_artist'])):
    # Option C edge cut, applied within the fold
    train_artists = {a for lst in artist_lists.iloc[train_idx] for a in lst}
    keep = [i for i in hold_idx
            if not any(a in train_artists for a in artist_lists.iloc[i])]
    dropped = len(hold_idx) - len(keep)
    # carve a validation slice out of train for early stopping
    rng = np.random.default_rng(mc.RANDOM_SEED + fold)
    shuffled = rng.permutation(train_idx)
    cut = int(0.85 * len(shuffled))
    fit_idx, es_idx = shuffled[:cut], shuffled[cut:]

    per_fold = {}
    for name in PAIR:
        c = cols_of[name]
        model = fit_xgb(X_all.iloc[fit_idx][c], y.iloc[fit_idx],
                        X_all.iloc[es_idx][c], y.iloc[es_idx])
        per_fold[name] = mc.regression_metrics(y.iloc[keep], model.predict(X_all.iloc[keep][c]))
    delta = per_fold['alignment_augmented']['r2'] - per_fold['control']['r2']
    rows.append({'fold': fold, 'n_holdout': len(keep), 'n_edge_cut': dropped,
                 'control_r2': per_fold['control']['r2'],
                 'alignment_augmented_r2': per_fold['alignment_augmented']['r2'],
                 'delta_r2': delta})
    print('fold %d  n=%6d (cut %5d)  control %+.5f  augmented %+.5f  Δ %+.5f'
          % (fold, len(keep), dropped, per_fold['control']['r2'],
             per_fold['alignment_augmented']['r2'], delta))

cv = pd.DataFrame(rows)
cv.to_csv(mc.artifact('robustness_cv.csv'), index=False)
signs = np.sign(cv['delta_r2'])
print()
print('ΔR2 across folds: mean %+.5f, std %.5f' % (cv['delta_r2'].mean(), cv['delta_r2'].std()))
print('sign consistent across all %d folds: %s' % (K, bool((signs == signs.iloc[0]).all())))

## 3. Hyperparameter robustness

Both models within each comparison always share identical settings, so this tests whether the direction and rough magnitude of the effect persist across reasonable configurations rather than appearing under only one.

In [ ]:
ALTERNATIVES = [
    {'label': 'selected (frozen)', 'params': {}},
    {'label': 'shallower, slower', 'params': {'max_depth': 4, 'learning_rate': 0.05}},
    {'label': 'deeper, faster', 'params': {'max_depth': 8, 'learning_rate': 0.1}},
    {'label': 'less stochastic', 'params': {'subsample': 1.0, 'colsample_bytree': 1.0}},
]

rows = []
for alt in ALTERNATIVES:
    per_alt = {}
    for name in PAIR:
        c = cols_of[name]
        model = fit_xgb(X_all.loc[tr, c], y[tr], X_all.loc[va, c], y[va], params=alt['params'])
        per_alt[name] = mc.regression_metrics(y[te], model.predict(X_all.loc[te, c]))
    delta = per_alt['alignment_augmented']['r2'] - per_alt['control']['r2']
    rows.append({'setting': alt['label'], 'params': json.dumps(alt['params']),
                 'control_r2': per_alt['control']['r2'],
                 'alignment_augmented_r2': per_alt['alignment_augmented']['r2'],
                 'delta_r2': delta})
    print('%-20s control %+.5f  augmented %+.5f  Δ %+.5f'
          % (alt['label'], per_alt['control']['r2'],
             per_alt['alignment_augmented']['r2'], delta))

hp = pd.DataFrame(rows)
hp.to_csv(mc.artifact('robustness_hyperparams.csv'), index=False)
hp_signs = np.sign(hp['delta_r2'])
print('\ndirection preserved across all settings: %s' % bool((hp_signs == hp_signs.iloc[0]).all()))

## 4. Cross-algorithm agreement, and the summary

The XGBoost-vs-Random Forest comparison was computed in notebook 7; it is restated here so Section 4.9.2's four checks land in one artifact.

In [ ]:
summary = {
    'observed_delta_r2_xgboost': central['xgboost']['delta_r2'],
    'bootstrap_ci': [central['xgboost']['ci_low'], central['xgboost']['ci_high']],
    'bootstrap_excludes_zero': central['xgboost']['ci_excludes_zero'],
    'seed_sensitivity': {
        'seeds': SEEDS,
        'r2_std_across_seeds': seed_sd,
        'mean_delta_r2': mean_delta,
        'delta_exceeds_seed_noise': bool(abs(mean_delta) > seed_sd),
        'sign_consistent': bool(np.all(np.sign(delta_spread) == np.sign(mean_delta))),
    },
    'cross_validation': {
        'k': K,
        'mean_delta_r2': float(cv['delta_r2'].mean()),
        'std_delta_r2': float(cv['delta_r2'].std()),
        'sign_consistent_across_folds': bool((signs == signs.iloc[0]).all()),
        'per_fold': cv.round(6).to_dict('records'),
    },
    'hyperparameter_robustness': {
        'direction_preserved': bool((hp_signs == hp_signs.iloc[0]).all()),
        'settings': hp.round(6).to_dict('records'),
    },
    'cross_algorithm': {
        'xgboost_delta_r2': central['xgboost']['delta_r2'],
        'random_forest_delta_r2': central['random_forest']['delta_r2'],
        'sign_agreement': central['cross_algorithm_sign_agreement'],
    },
}
print(json.dumps(summary, indent=2, default=str))
print()
print(mc.save_json(summary, 'robustness_summary.json'))

checks = {
    'seed sensitivity': summary['seed_sensitivity']['delta_exceeds_seed_noise'],
    'CV stability (sign)': summary['cross_validation']['sign_consistent_across_folds'],
    'hyperparameter robustness': summary['hyperparameter_robustness']['direction_preserved'],
    'cross-algorithm agreement': summary['cross_algorithm']['sign_agreement'],
}
print('\n=== Section 4.9.2 scorecard ===')
for k, v in checks.items():
    print('  %-28s %s' % (k, 'PASS' if v else 'FAIL'))

---
### Interpreting the scorecard

These are **comparative** criteria, not binary gates like the Section 4.9.1 verification checks. A FAIL does not invalidate the pipeline -- it constrains what can be claimed:

- **Seed sensitivity FAIL** is the most serious. It means the observed difference is inside the noise floor of the training procedure, and Section 4.9.2 requires reporting it as such even if the bootstrap interval excluded zero.
- **CV sign inconsistency** means the effect is a property of the one partition rather than of the feature set.
- **Hyperparameter or cross-algorithm disagreement** means the effect is tied to one configuration or one algorithm's training dynamics.

Report all four outcomes in the chapter whichever way they land. The study's credibility rests on the checks having been run and reported, not on them all passing.

Remaining work outside these notebooks: freeze `xgboost__alignment_augmented.joblib` for the demo app's `POST /predict` path (REQ-4.4-8, Objective 6), which does not exist yet.